In [1]:
import pandas as pd
import re

# 1. Load the dataset (assuming fresh start in the new notebook)
df = pd.read_csv("../data/raw/realdonaldtrump.csv")

# 2. Define a robust text cleaning function
def clean_tweet(text):
    # Handle any missing values by returning an empty string
    if not isinstance(text, str):
        return ""

    # Lowercase the text
    text = text.lower()

    # Remove URLs (http, https, www)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove user mentions (e.g., @realdonaldtrump)
    text = re.sub(r'@\w+', '', text)

    # Remove special characters, numbers, and emojis
    # (Keeps only standard English alphabetical characters and spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra whitespaces that might have appeared after removing words
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("--- TEXT CLEANING IN PROGRESS ---")

# 3. Apply the function to create a NEW column
# We keep the original 'content' column safe for Person 2 (Feature Engineering)
df['cleaned_content'] = df['content'].apply(clean_tweet)

# 4. Display the results to compare original text with the cleaned version
print("Original vs Cleaned Tweets (First 5 rows):")
# Use pandas display functionality to show it nicely in Jupyter
display(df[['content', 'cleaned_content']].head())

--- TEXT CLEANING IN PROGRESS ---
Original vs Cleaned Tweets (First 5 rows):


,content,cleaned_content
0,Be sure to tune in and watch Donald Trump on L...,be sure to tune in and watch donald trump on l...
1,Donald Trump will be appearing on The View tom...,donald trump will be appearing on the view tom...
2,Donald Trump reads Top Ten Financial Tips on L...,donald trump reads top ten financial tips on l...
3,New Blog Post: Celebrity Apprentice Finale and...,new blog post celebrity apprentice finale and ...
4,"""My persona will never be that of a wallflower...",my persona will never be that of a wallflower ...


In [3]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# 1. Download required NLTK datasets (run this once)
# 'punkt' is for tokenization, 'wordnet' for lemmatization
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# 2. Initialize the tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# 3. Define the NLP processing function
def process_nlp(text):
    # Fallback for empty values
    if not text or not isinstance(text, str):
        return ""

    # A. Tokenization: Split text into a list of individual words
    tokens = word_tokenize(text)

    # B. Stopwords removal & C. Lemmatization
    # We also remove words that are just 1 character long (e.g., isolated 's' or 't')
    processed_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    # D. Rejoin into a single string
    # (Models like TF-IDF used by Person 2 require string format, not lists)
    return " ".join(processed_tokens)

print("--- NLP PROCESSING IN PROGRESS ---")

# 4. Apply the function to create the final processed column
df['processed_content'] = df['cleaned_content'].apply(process_nlp)

# 5. Handle missing data in other columns (Crucial for Person 2's Feature Engineering)
# We fill NaN values in 'hashtags' and 'mentions' with empty strings
df['hashtags'] = df['hashtags'].fillna("")
df['mentions'] = df['mentions'].fillna("")

# 6. Display the full transformation pipeline
print("Pipeline Check (Original -> Cleaned -> Processed):")
display(df[['content', 'cleaned_content', 'processed_content']].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alicj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alicj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alicj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alicj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


--- NLP PROCESSING IN PROGRESS ---
Pipeline Check (Original -> Cleaned -> Processed):


,content,cleaned_content,processed_content
0,Be sure to tune in and watch Donald Trump on L...,be sure to tune in and watch donald trump on l...,sure tune watch donald trump late night david ...
1,Donald Trump will be appearing on The View tom...,donald trump will be appearing on the view tom...,donald trump appearing view tomorrow morning d...
2,Donald Trump reads Top Ten Financial Tips on L...,donald trump reads top ten financial tips on l...,donald trump read top ten financial tip late s...
3,New Blog Post: Celebrity Apprentice Finale and...,new blog post celebrity apprentice finale and ...,new blog post celebrity apprentice finale less...
4,"""My persona will never be that of a wallflower...",my persona will never be that of a wallflower ...,persona never wallflower rather build wall cli...
